In [39]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

class SparkConfig :
    def creat_sparksession() :
        spark = SparkSession.builder.config("spark.driver.memory" , "8g") \
                            .appName("clean data") \
                            .master("local[*]") \
                            .getOrCreate()
        spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", "file:///")
        return spark

In [40]:
spark = SparkConfig.creat_sparksession()

In [41]:
class DataLoader :
    def __init__(selt) :
        selt.spark = SparkConfig.create_sparksession() 
    def clean_works() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            works = spark.read.format("parquet").load(file) 
            
            works = works.withColumn("cover_id" , col("cover_id").cast("long"))
            for c in works.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in works.columns :
                    works = works.withColumn(c , lit(None).cast("string"))
            list_file.append(works)
        works = reduce(lambda a,b : a.union(b) , list_file)
        return works

    def clean_editions() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/editions/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            editions = spark.read.format("parquet").load(file) 
            if "number_of_pages" in editions.columns :
                editions = editions.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            for c in editions.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in editions.columns :
                    editions = editions.withColumn(c , lit(None).cast("string"))
            list_file.append(editions)
        editions = reduce(lambda a,b : a.union(b) , list_file)
        editions.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/editions/")
        return editions 
    
    def clean_works() :        
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/authors/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        for file in files :
            authors = spark.read.format("parquet").load(file) 
            if "number_of_pages" in authors.columns :
                authors = authors.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            list_file.append(authors)
        authors = reduce(lambda a,b : a.unionByName(b , allowMissingColumns = True) , list_file)
        authors.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/authors/")
        return authors

In [42]:
spark

In [43]:
works = spark.read.parquet("data/works.parquet")

In [44]:
editions = spark.read.parquet("data/editions.parquet")

In [45]:
authors = spark.read.parquet("data/authors.parquet")

In [46]:
# Chọn các cột cần để phân tích
works = works.select("key", "title", "first_publish_year", "edition_count", "subject", "authors" )

In [47]:
works.printSchema()

root
 |-- key: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_publish_year: double (nullable = true)
 |-- edition_count: long (nullable = true)
 |-- subject: string (nullable = true)
 |-- authors: string (nullable = true)



In [48]:
works.select([count(when (col(c).isNull() , c)).alias(c) for c in works.columns ]).show()

+---+-----+------------------+-------------+-------+-------+
|key|title|first_publish_year|edition_count|subject|authors|
+---+-----+------------------+-------------+-------+-------+
|  0|    0|               145|            0|      0|      0|
+---+-----+------------------+-------------+-------+-------+



In [49]:
works.show()

+-----------------+--------------------+------------------+-------------+--------------------+--------------------+
|              key|               title|first_publish_year|edition_count|             subject|             authors|
+-----------------+--------------------+------------------+-------------+--------------------+--------------------+
|  /works/OL21177W|   Wuthering Heights|            1846.0|         2886|["form:novel", "g...|[{"key": "/author...|
|  /works/OL66513W|                Emma|            1815.0|         2263|["Social life and...|[{"key": "/author...|
|  /works/OL66562W|Sense and Sensibi...|            1811.0|         2090|["Fiction, Romanc...|[{"key": "/author...|
|  /works/OL29983W|        Little Women|            1848.0|         1888|["Romans", "Jeune...|[{"key": "/author...|
| /works/OL267096W|       Анна Каренина|            1876.0|         1314|["Fiction", "Adul...|[{"key": "/author...|
|/works/OL1095427W|           Jane Eyre|            1847.0|         1170

In [50]:
# Làm sạch cột key và first_pubnlish_year 
work_key = works.withColumn("key" , regexp_extract(col("key") , r"[A-Z]+\d+[A-Z]", 0)) \
                .withColumnRenamed("key", "work_id") \
                .withColumn("first_publish_year" , col("first_publish_year").cast("int")) # chuyển đổi kiểu dữ liệu cột first_publish_year

In [51]:
# Xử lý giá trị missing value bằng cách lấy trung vị
median = work_key.approxQuantile("first_publish_year", [0.5], 0.01)[0]
work_key = work_key.fillna({"first_publish_year" : int(median)})

In [52]:
# Tạo ra bảng dim_work 
dim_work = work_key.select("work_id", "title", "first_publish_year", "edition_count").dropDuplicates()

In [71]:
# Explode cột subject 
work_explode_sub = work_key.withColumn("subject" , regexp_replace("subject" , r"[\]\[]", "")) \
        .withColumn("subject", explode(split("subject" , ",")))

In [ ]:
# Làm sạch cột subject
work_explode_sub = work_explode_sub.withColumn("subject", regexp_replace("subject", r'form:|genre:|"', ""))

In [ ]:
work_subject = 

+--------+-----------------+------------------+-------------+--------------------+--------------------+
| work_id|            title|first_publish_year|edition_count|             subject|             authors|
+--------+-----------------+------------------+-------------+--------------------+--------------------+
|OL21177W|Wuthering Heights|              1846|         2886|               novel|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|             tragedy|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|              gothic|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886| British and iris...|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|  Children's fiction|[{"key": "/author...|
+--------+-----------------+------------------+-------------+--------------------+--------------------+
only showing top 5 rows


In [ ]:
# Explode cột authors 
work_explode_au = work_key.withColumn("authors" , regexp_replace("authors" , r"[\]\[]", "")) \
        .withColumn("authors", explode(split("authors" , ",")))

In [ ]:
work_explode_au.show(5)

+--------+--------------------+------------------+-------------+--------------------+--------------------+
| work_id|               title|first_publish_year|edition_count|             subject|             authors|
+--------+--------------------+------------------+-------------+--------------------+--------------------+
|OL21177W|   Wuthering Heights|              1846|         2886|["form:novel", "g...|{"key": "/authors...|
|OL21177W|   Wuthering Heights|              1846|         2886|["form:novel", "g...| "name": "Emily B...|
|OL66513W|                Emma|              1815|         2263|["Social life and...|{"key": "/authors...|
|OL66513W|                Emma|              1815|         2263|["Social life and...| "name": "Jane Au...|
|OL66562W|Sense and Sensibi...|              1811|         2090|["Fiction, Romanc...|{"key": "/authors...|
+--------+--------------------+------------------+-------------+--------------------+--------------------+
only showing top 5 rows


+--------+-----------------+------------------+-------------+--------------------+--------------------+
| work_id|            title|first_publish_year|edition_count|             subject|             authors|
+--------+-----------------+------------------+-------------+--------------------+--------------------+
|OL21177W|Wuthering Heights|              1846|         2886|               novel|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|             tragedy|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|              gothic|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886| British and iris...|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|  Children's fiction|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|     Classic fiction|[{"key": "/author...|
|OL21177W|Wuthering Heights|              1846|         2886|  C

In [ ]:
work_key.show(5)

+---------+--------------------+------------------+-------------+--------------------+--------------------+
|  work_id|               title|first_publish_year|edition_count|             subject|             authors|
+---------+--------------------+------------------+-------------+--------------------+--------------------+
| OL21177W|   Wuthering Heights|              1846|         2886|["form:novel", "g...|[{"key": "/author...|
| OL66513W|                Emma|              1815|         2263|["Social life and...|[{"key": "/author...|
| OL66562W|Sense and Sensibi...|              1811|         2090|["Fiction, Romanc...|[{"key": "/author...|
| OL29983W|        Little Women|              1848|         1888|["Romans", "Jeune...|[{"key": "/author...|
|OL267096W|       Анна Каренина|              1876|         1314|["Fiction", "Adul...|[{"key": "/author...|
+---------+--------------------+------------------+-------------+--------------------+--------------------+
only showing top 5 rows


## Edition

In [ ]:
editions = editions.select("key", "works", "title", "publish_date", "publishers", "languages", "number_of_pages", "isbn_13")

In [ ]:
editions.show(5)

+------------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|               key|               works|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+------------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|/books/OL61212055M|[{"key": "/works/...|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|/books/OL56513356M|[{"key": "/works/...| Hauts de Hurle-Vent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            356|["9781974501502"]|
|/books/OL56523458M|[{"key": "/works/...|  Hauts de Hurlevent|        2017|["CreateSpace Ind...|[{"key": "/langua...|            368|["9781981584758"]|
|/books/OL56296338M|[{"key": "/works/...| Hauts de Hurle-Vent|        2017|["CreateSpace

## Authors

In [ ]:
authors = authors.select("key", "name", "birth_date")

In [ ]:
authors.show(10)

+--------------------+-----------------+------------+
|                 key|             name|  birth_date|
+--------------------+-----------------+------------+
|   /authors/OL24529A|    Emily Brontë|30 July 1818|
| /authors/OL9082707A|       Sir Angels|        NULL|
|/authors/OL10885515A|édéric Delebecque|        NULL|
|/authors/OL10922641A|     Gisela Etzel|        NULL|
|   /authors/OL24529A|    Emily Brontë|30 July 1818|
| /authors/OL2957071A|    Tanya Landman|        NULL|
| /authors/OL9082707A|       Sir Angels|        NULL|
|  /authors/OL342487A|      Anne Brontë|        1820|
|/authors/OL10885515A|édéric Delebecque|        NULL|
| /authors/OL4696201A| Barnett Freedman|        1901|
+--------------------+-----------------+------------+
only showing top 10 rows


In [ ]:
works.withColumn("subject" , col("subject").cast("array<string>")) \
     .withColumn("subject_exp" , explode(col("subject"))).select("subject_exp").show(5)

{"ts": "2026-04-29 07:34:27.643", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve \"subject\" due to data type mismatch: cannot cast \"STRING\" to \"ARRAY<STRING>\". SQLSTATE: 42K09", "context": {"file": "line 1 in cell [21]", "line": "", "fragment": "cast", "errorClass": "DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o55.withColumn.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve \"subject\" due to data type mismatch: cannot cast \"STRING\" to \"ARRAY<STRING>\". SQLSTATE: 42K09;\n'Project [key#0, title#1, first_publish_year#11, edition_count#2L, cast(subject#5 as array<string>) AS subject#298, authors#10]\n+- Project [key#0, title#1, first_publish_year#11, edition_count#2L, subject#5, authors#10]\n   +- Relation [key#0,title#1,edition_count#2L,cover_id#3L,cover_edition_key#4,su

AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "subject" due to data type mismatch: cannot cast "STRING" to "ARRAY<STRING>". SQLSTATE: 42K09;
'Project [key#0, title#1, first_publish_year#11, edition_count#2L, cast(subject#5 as array<string>) AS subject#298, authors#10]
+- Project [key#0, title#1, first_publish_year#11, edition_count#2L, subject#5, authors#10]
   +- Relation [key#0,title#1,edition_count#2L,cover_id#3L,cover_edition_key#4,subject#5,ia_collection#6,printdisabled#7,lending_edition#8,lending_identifier#9,authors#10,first_publish_year#11,ia#12,public_scan#13,has_fulltext#14,availability#15] parquet


In [ ]:
works.count()

31901

In [ ]:
editions.count()

224540

In [ ]:
editions.select([count(when (col(c).isNull() , c)).alias(c) for c in editions.columns ]).show()

+---+-----+-----+------------+----------+---------+---------------+-------+
|key|works|title|publish_date|publishers|languages|number_of_pages|isbn_13|
+---+-----+-----+------------+----------+---------+---------------+-------+
|  0|    0|    9|        2957|      1978|    27963|         116486|  59642|
+---+-----+-----+------------+----------+---------+---------------+-------+



In [ ]:
authors.select([count(when (col(c).isNull() , c)).alias(c) for c in authors.columns ]).show()

+---+----+----------+
|key|name|birth_date|
+---+----+----------+
|  0|1351|    114660|
+---+----+----------+



In [ ]:
authors.count()

125500